# MAAS Deep Policy Notebook

            This notebook trains a **MAAS-specific deep temporal policy model** over structured prenatal observations and short daily-history sequences.

            It is designed to be directly usable with the MAAS project:
            - it reads the MAAS `prenatal.db` when available
            - it uses the same observation semantics as `environment.py`
            - it predicts the same action schema used by the app: `condition`, `urgency`, `rationale`
            - it can export a checkpoint that can later be wired into the backend

            The model is intentionally more ambitious than the lightweight rule baselines:
            - Transformer encoder over 3-step maternal history
            - gated static/temporal feature fusion
            - multi-task heads for condition, urgency, danger escalation, and latent complication signals


## 1. Install Dependencies

            If you run this in Colab or a fresh Jupyter environment, install the core packages first.


In [ ]:
%pip install torch numpy pillow sqlalchemy pydantic fastapi -q
            print("Dependencies ready")


## 2. Point Python at the MAAS repo


In [ ]:
import sys
            from pathlib import Path

            REPO_ROOT = Path.cwd()
            if str(REPO_ROOT) not in sys.path:
                sys.path.insert(0, str(REPO_ROOT))

            print("Repo root:", REPO_ROOT)
            print("prenatal.db exists:", (REPO_ROOT / "prenatal.db").exists())


## 3. Import the MAAS deep policy stack


In [ ]:
from maas_deep_policy import (
                load_seed_cases,
                predict_for_user_id,
                render_training_curve,
                save_checkpoint,
                save_history,
                train_model,
            )

            seed_cases = load_seed_cases(REPO_ROOT / "prenatal.db")
            print("Loaded seed cases:", len(seed_cases))
            print("Example sources:", [seed.source_id for seed in seed_cases[:5]])


## 4. Configure a MAAS experiment

            Increase `num_samples` and `epochs` when you move to a GPU runtime.


In [ ]:
EXPERIMENT = {
                "num_samples": 4096,
                "epochs": 14,
                "batch_size": 128,
                "learning_rate": 0.002,
                "weight_decay": 1e-4,
                "seed": 42,
                "db_path": str(REPO_ROOT / "prenatal.db"),
            }
            EXPERIMENT


## 5. Train the deep policy model


In [ ]:
artifacts = train_model(**EXPERIMENT)
            history = artifacts["history"]
            history[-1]


## 6. Save checkpoint, metrics, and training-curve image


In [ ]:
OUTPUT_DIR = REPO_ROOT / "artifacts" / "maas_deep_policy_notebook"
            OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

            checkpoint_path = save_checkpoint(
                model=artifacts["model"],
                region_to_idx=artifacts["region_to_idx"],
                output_dir=OUTPUT_DIR,
                config=artifacts["config"],
            )
            history_path = save_history(history, OUTPUT_DIR / "training_history.json")
            curve_path = render_training_curve(history, OUTPUT_DIR / "training_curve.png")

            print("Checkpoint:", checkpoint_path)
            print("History:", history_path)
            print("Curve:", curve_path)


## 7. Display the training curve


In [ ]:
from IPython.display import Image, display

            display(Image(filename=str(curve_path)))


## 8. Run a real MAAS-style prediction

            This uses a real patient from `prenatal.db` and returns the same schema the MAAS backend expects.


In [ ]:
prediction = predict_for_user_id(
                user_id=2,
                checkpoint_path=checkpoint_path,
                db_path=REPO_ROOT / "prenatal.db",
            )
            prediction


## 9. What this model gives MAAS

            - A trainable structured policy instead of only rules
            - A checkpoint that can be loaded for backend inference
            - A richer representation over short maternal history sequences
            - A multi-head design that can support escalation confidence and latent-risk monitoring
